# Word Embedding을 활용한 단어 찾기 게임

- Level 1. [Semantle](https://semantle.com/) 흉내내기
- Level 2. Semantle의 한국어 버전을 만들어보기
- Level 3. 단어 `A:B=C:?` 여기서 ? 맞추기 ([한국어 임베딩 확인](https://word2vec.kr/search/) + Semantle)

### Level 1

In [39]:
from lxml import etree

f = open('../data/ted_en.xml','r',encoding='utf-8')
xml = etree.parse(f)

contents = xml.xpath('//content/text()')
corpus = '\n'.join(contents)
print(len(corpus))



24222849


In [40]:
import re
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

# 정규식을 이용해 (Laughter), (Applause) 등 키워드 제거
corpus = re.sub(r'\([^)]*\)', '', corpus)

sentences = sent_tokenize(corpus)

preprocessed_sentences = []
en_stopwords = stopwords.words('english')

for sentence in sentences:
    sentence = sentence.lower()
    sentence = re.sub(r'[^a-z0-9]', ' ', sentence)
    tokens = word_tokenize(sentence)
    tokens = [token for token in tokens if token not in en_stopwords]
    preprocessed_sentences.append(tokens)

In [35]:
from gensim.models import Word2Vec
w2v_e_model = Word2Vec(
    sentences=preprocessed_sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=0
)

In [41]:
from gensim.models import FastText

ft_model = FastText(
    sentences=preprocessed_sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=0
)


In [44]:
answer = 'strawberry'
user_answer = input()
if user_answer in w2v_e_model.wv:
    similar = w2v_e_model.wv.similarity(answer, user_answer)
    if similar == 1:
        print('단어를 맞췄습니다!!')
    else :
        print('단어 : ', user_answer,'유사도 : ', similar)
else :
    print('단어 : ', user_answer, '유사도 : 0')

 apple


단어 :  apple 유사도 :  0.7015842


### Level 2

In [45]:
import json
import pandas as pd
data_list = []
with open('../data/train_original.json', 'r', encoding='utf-8') as f:
        text_json = json.load(f, strict=False)
sentences = []

for doc in text_json['documents']:
    for text in doc['text']:
        for sen in text:
            sentences.append(sen['sentence'])

In [15]:
# 전처리
from konlpy.tag import Okt

okt = Okt()

sent_nouns = []

for i in sentences[:100000]:
    nouns = okt.nouns(i)

    nouns = [n for n in nouns if len(n)>1]
    sent_nouns.append(nouns)

print(sent_nouns[:10])

[['이명박', '대통령', '어제', '그룹', '총수', '시대', '요구', '역시', '총수', '변화', '조짐', '생각', '총수', '직접', '관심', '전파', '긍정', '평가'], ['언뜻', '보아', '무슨', '불분명', '하나', '대통령', '지난', '연설', '정몽준', '의원', '정몽구', '현대차', '회장', '각각', '기부', '사실', '공생', '발전', '이란', '화두', '금방', '짐작', '간다'], ['다른', '그룹', '총수', '은근'], ['대통령', '기부', '대한', '후속', '선언', '총수', '사회', '공헌', '방안', '불만', '표시', '후문'], ['최근', '미국', '프랑스', '벨기에', '부유세', '거론', '독일', '한시', '자산', '마련', '논의'], ['흐름', '한국', '글로벌', '시대', '특징'], ['항간', '이번', '삼성', '비롯', '몇몇', '그룹', '노블레스', '오블리주', '방안', '준비', '대통령', '강요', '포퓰리즘', '압박', '자발', '문화', '효과'], ['재계', '방안', '마련', '정치권', '여론', '압박', '시간'], ['국가채무', '문제', '글로벌', '경기', '침체', '우려', '상황', '기업', '생존', '부담', '때문'], ['전경련', '그룹', '올해', '고용', '투자', '선물', '준비']]


In [24]:
text_df = pd.DataFrame(sent_nouns)
text_df.to_csv('../data/word_game_sentence.csv', index = False)

In [46]:
from gensim.models import Word2Vec
w2v_model = Word2Vec(
    sentences=sent_nouns,
    vector_size=100,
    window=5,
    min_count=5,
    sg=0
)

In [47]:
from gensim.models import FastText

ft_model = FastText(
    sentences=sent_nouns,
    vector_size=100,
    window=5,
    min_count=5,
    sg=0
)

In [49]:
answer = '선물'
user_answer = input()
if user_answer in w2v_model.wv:
    similar = w2v_model.wv.similarity(answer, user_answer)
    if similar == 1:
        print('단어를 맞췄습니다!!')
    else :
        print('단어 : ', user_answer,', 유사도 : ', similar)
else :
    print('단어 : ', user_answer, '유사도 : 0')

 유기농


단어 :  유기농 , 유사도 :  0.68320966


### LEVEL 3

In [69]:
result = w2v_model.wv.most_similar(positive=['미국', '뉴욕'], negative=['대한민국'])
print(result)

[('유럽연합', 0.7232086658477783), ('이란', 0.7205671668052673), ('통화스왑', 0.706601619720459), ('대만', 0.7059775590896606), ('캐나다', 0.7056774497032166), ('비트코인', 0.6970340013504028), ('중국산', 0.6963621973991394), ('중앙은행', 0.69371098279953), ('파운드', 0.6884492039680481), ('쇠고기', 0.6822715401649475)]
